In [ ]:
%load_ext autoreload
%autoreload 2

In [6]:
from brain_image.model.comm.comm import CoMM
from brain_image.model.comm.encoders import AlexNetEncoder
from brain_image.model.comm.input_adapters import FeaturesInputAdapter, PatchedInputAdapter
from brain_image.model.comm.mmfusion import MMFusion
from brain_image.model.eeg_encoder.atms import AtmsEEGEncoder, AtmsEEGEncoderConfig


comm = CoMM(
    encoder=MMFusion(
        encoders=[ # Symmetric visual encoders
            AlexNetEncoder(latent_dim=512, global_pool=""), 
            AtmsEEGEncoder(AtmsEEGEncoderConfig(d_channels=63, d_output=512))
        ], 
        input_adapters=[ # Pach adapters for multimodal fusion
            PatchedInputAdapter(num_channels=256, stride_level=1, patch_size_full=1, dim_tokens=512, image_size=6),
            #PatchedInputAdapter(num_channels=256, stride_level=1, patch_size_full=1, dim_tokens=512, image_size=6)
            FeaturesInputAdapter(512, 512)
        ],
        embed_dim=512
    ),
    projection=CoMM._build_mlp(512, 512, 256),
    optim_kwargs=dict(lr=3e-4, weight_decay=1e-4),
    loss_kwargs=dict(temperature=0.1)
)

In [ ]:
from torch.utils.data import DataLoader
from pathlib import Path

from brain_image.data.things_eeg2_dataset import ThingsEEG2Dataset, ThingsEEG2DatasetConfig


things_path = Path("data/things-eeg2")
dataset_config = ThingsEEG2DatasetConfig(subs=[1], num_workers=0)
dataset = ThingsEEG2Dataset(
    dataset_config, 
    split="test", 
)
loader = DataLoader(dataset, batch_size=4, num_workers=0)

In [12]:
from brain_image.data.data import batch_load_images, load_image_from_path


ex_batch = next(iter(loader))
imgs = batch_load_images(ex_batch["img_path"])
eeg = ex_batch["eeg_data"]

In [ ]:
comm.forward([img, eeg], [img, eeg])

RuntimeError: mat1 and mat2 must have the same dtype, but got Double and Float